# Realistic Human Face Generation (DCGAN vs. WGAN-GP)
## Step 0: Setup and Kaggle Prerequisites

In this notebook, I develop and train two Generative Adversarial Network (GAN) architectures from scratch using PyTorch:
1. **DCGAN (Deep Convolutional GAN)**
2. **WGAN-GP (Wasserstein GAN with Gradient Penalty)**

### Kaggle Dataset & GPU Instructions:
1. In Kaggle, click **( + Add Data )**, search for **`celeba-dataset`** (by Jessica Li), and click **Add**.
2. Make sure the Accelerator is set to **GPU T4 x2** (or GPU T4 / P100).
3. **How I optimized the training speed:**
   - Set `BATCH_SIZE = 256` and `N_CRITIC = 1` for WGAN-GP so each epoch finishes in ~4-5 minutes.
   - Enabled PyTorch Automatic Mixed Precision (AMP) for faster FP16 convolutions.
   - Kept the WGAN-GP Gradient Penalty in FP32 so gradients never overflow.
   - Disabled persistent workers and cleared image references to prevent system RAM leaks.


In [ ]:
!pip install -q psutil tqdm pillow


## Step 1: Import Libraries and Set Hardware Flags

Here I import the required PyTorch modules, torchvision transforms, and utility libraries.
I also enable `cuDNN benchmark` and `TF32` to get the fastest training performance on NVIDIA Ampere/Turing GPUs.


In [ ]:
import os, sys, gc, json, ctypes, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import psutil
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torchvision.io import read_image, ImageReadMode
from torch.utils.data import DataLoader, Dataset
from PIL import Image as PILImage

import matplotlib
matplotlib.use('Agg')          # Non-interactive backend to prevent Jupyter GUI memory leaks
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, Image, display

# Hardware acceleration and cuDNN flags
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Project hyperparameters
BATCH_SIZE        = 256    # Set to 256 to double throughput on Kaggle T4 (~5 mins/epoch for WGAN-GP)
IMAGE_SIZE        = 64
NZ                = 100    # Latent vector dimension
NGF               = 64     # Generator feature map depth
NDF               = 64     # Discriminator feature map depth
NUM_EPOCHS        = 50
LR_DCGAN          = 0.0002
BETA1             = 0.5

# WGAN-GP hyperparameters
LAMBDA_GP         = 10
N_CRITIC          = 1      # Set to 1 for ultra-fast WGAN-GP training (~4-5 mins/epoch on T4 64x64)
LR_WGAN           = 0.0002

# Training optimization flags
USE_AMP           = True   # Automatic Mixed Precision (FP16 autocast + GradScaler)
USE_CHANNELS_LAST = False  # Recommended False with torch.compile to avoid Inductor ConvTranspose2d stride bugs
USE_COMPILE       = True   # PyTorch 2.x model compilation
USE_MULTI_GPU     = False  # Recommended False for 64x64 on T4 to avoid DataParallel overhead

# GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ngpu   = torch.cuda.device_count()
print(f"Device : {device}  |  GPUs Available : {ngpu} | Multi-GPU Wrapped : {USE_MULTI_GPU and ngpu > 1}")

# Output directories
OUTPUT_DIR  = Path("/kaggle/working/outputs")
MODELS_DIR  = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
IMAGES_DIR  = OUTPUT_DIR / "images"
CKPT_DIR    = OUTPUT_DIR / "checkpoints"

for d in [MODELS_DIR, METRICS_DIR, IMAGES_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Async File I/O Executor for Non-Blocking Image Saving
img_writer = ThreadPoolExecutor(max_workers=2)

def save_image_async(tensor_grid, filepath):
    """Save image grid to disk asynchronously without blocking the training loop."""
    def _write():
        vutils.save_image(tensor_grid, str(filepath), padding=2, normalize=True)
    img_writer.submit(_write)

def get_system_stats():
    """Return precise memory and GPU utilization report."""
    ram = psutil.virtual_memory()
    rss = psutil.Process().memory_info().rss / (1024 ** 2)
    gpu_stats = []
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            alloc = torch.cuda.memory_allocated(idx) / (1024 ** 2)
            res   = torch.cuda.memory_reserved(idx) / (1024 ** 2)
            gpu_stats.append(f"GPU{idx}: Alloc={alloc:.0f}MB/Res={res:.0f}MB")
    gpu_str = " | ".join(gpu_stats) if gpu_stats else "No GPU"
    return f"RSS: {rss:.0f}MB | RAM: {ram.used/(1024**3):.1f}/{ram.total/(1024**3):.1f}GB ({ram.percent}%) | {gpu_str}"

print("System Diagnostics Helper Ready.")
print(get_system_stats())


## Step 2: Dataset Preprocessing and DataLoader

In this section, I load the face dataset (CelebA or FFHQ sample faces) and preprocess the images:
- Crop and resize images to `64x64`.
- Normalize pixel values from `[0, 255]` to `[-1, 1]` (`transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))`).
- Create a memory-safe PyTorch `DataLoader` with `BATCH_SIZE = 256`.


In [ ]:
class CelebADiskDataset(Dataset):
    """Reads JPEG files directly into C++ Tensors using torchvision.io.read_image."""
    EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    def __init__(self, root: str, transform=None):
        self.root      = Path(root)
        self.transform = transform
        self.paths     = sorted([
            str(p) for p in self.root.rglob('*')
            if p.suffix.lower() in self.EXTENSIONS
        ])
        if len(self.paths) == 0:
            raise RuntimeError(f"No image files found in {self.root}.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path_str = self.paths[idx]
        
        # Hint Linux kernel to drop page cache for this file after reading
        if hasattr(os, 'posix_fadvise') and hasattr(os, 'POSIX_FADV_DONTNEED'):
            try:
                fd = os.open(path_str, os.O_RDONLY)
                os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                os.close(fd)
            except Exception:
                pass

        img = read_image(path_str, mode=ImageReadMode.RGB)
        # Prevent holding internal torchvision storage reference
        img = img.contiguous().clone()
        if self.transform:
            img = self.transform(img)
        return img


# ── Locate CelebA Dataset Path ──────────────────────────────
POSSIBLE_PATHS = [
    "/kaggle/input/datasets/jessicali9530/celeba-dataset",
    "/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba",
    "/kaggle/input/celeba-dataset/img_align_celeba",
    "/kaggle/input/celeba-dataset",
]

CELEBA_ROOT = None
for p in POSSIBLE_PATHS:
    if Path(p).exists():
        CELEBA_ROOT = p
        break

if CELEBA_ROOT is None:
    CELEBA_ROOT = "/kaggle/input"

print(f"Dataset root: {CELEBA_ROOT}")

# ── Tensor Transformation Pipeline ──────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    transforms.ConvertImageDtype(torch.float),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = CelebADiskDataset(CELEBA_ROOT, transform=transform)

# Optimal DataLoader Settings for Kaggle (persistent_workers=False prevents RAM bloat)
NUM_WORKERS = 4
dataloader = DataLoader(
    dataset,
    batch_size          = BATCH_SIZE,
    shuffle             = True,
    num_workers         = NUM_WORKERS,
    pin_memory          = True,
    persistent_workers  = False, # Recommended False to avoid RAM growth across Epochs
    prefetch_factor     = 2 if NUM_WORKERS > 0 else None,
    drop_last           = True,
)

fixed_noise    = torch.randn(64, NZ, 1, 1, device=device)
real_batch_cpu = next(iter(dataloader)).cpu()

print(f"Total Dataset Images : {len(dataset):,}")
print(f"Batches per Epoch    : {len(dataloader):,} (Batch Size = {BATCH_SIZE})")
print(f"DataLoader Workers   : {NUM_WORKERS} (pin_memory=True, persistent_workers=False)")


## Step 3: Build Generator and Discriminator Architectures

I build custom deep convolutional models for both GANs:
- **DCGAN Generator & Discriminator**: Standard convolutional layers with `BatchNorm2d`.
- **WGAN-GP Generator & Critic**: The Critic uses `InstanceNorm2d` instead of BatchNorm because standard BatchNorm violates the Lipschitz continuity requirement in WGAN-GP.


In [ ]:
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Real CelebA Faces (64 Sample Grid)")
plt.imshow(np.transpose(
    vutils.make_grid(real_batch_cpu[:64], padding=2, normalize=True).numpy(), (1, 2, 0)
))
plt.savefig(str(IMAGES_DIR / "real_faces.png"), bbox_inches='tight')
plt.close('all')
display(Image(filename=str(IMAGES_DIR / "real_faces.png")))


## Step 4: Discriminator / Critic Helper Functions

Here I define helper functions to train the Discriminator (for DCGAN) and the Critic with Gradient Penalty (for WGAN-GP).


In [ ]:
def weights_init(m):
    cls = m.__class__.__name__
    if 'Conv' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class DCGAN_Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(NZ,    NGF*8, 4, 1, 0, bias=False), nn.BatchNorm2d(NGF*8), nn.ReLU(False),
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF*4), nn.ReLU(False),
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF*2), nn.ReLU(False),
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False), nn.BatchNorm2d(NGF),   nn.ReLU(False),
            nn.ConvTranspose2d(NGF,   3,     4, 2, 1, bias=False), nn.Tanh(),
        )
    def forward(self, x): return self.main(x)


class DCGAN_Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        SN = nn.utils.spectral_norm
        self.main = nn.Sequential(
            SN(nn.Conv2d(3,     NDF,   4, 2, 1, bias=False)),                           nn.LeakyReLU(0.2, inplace=False),
            SN(nn.Conv2d(NDF,   NDF*2, 4, 2, 1, bias=False)), nn.BatchNorm2d(NDF*2),   nn.LeakyReLU(0.2, inplace=False),
            SN(nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False)), nn.BatchNorm2d(NDF*4),   nn.LeakyReLU(0.2, inplace=False),
            SN(nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False)), nn.BatchNorm2d(NDF*8),   nn.LeakyReLU(0.2, inplace=False),
            SN(nn.Conv2d(NDF*8, 1,     4, 1, 0, bias=False)), # No Sigmoid for BCEWithLogitsLoss AMP safety!
        )
    def forward(self, x): return self.main(x)


netG_DC = DCGAN_Generator().to(device)
netD_DC = DCGAN_Discriminator().to(device)

if USE_CHANNELS_LAST:
    # Only convert Discriminator (Conv2d); ConvTranspose2d in Generator has Inductor stride bugs
    netD_DC = netD_DC.to(memory_format=torch.channels_last)

if USE_MULTI_GPU and ngpu > 1:
    netG_DC = nn.DataParallel(netG_DC, list(range(ngpu)))
    netD_DC = nn.DataParallel(netD_DC, list(range(ngpu)))

netG_DC.apply(weights_init)
netD_DC.apply(weights_init)

# PyTorch 2.x Model Compilation
if USE_COMPILE and hasattr(torch, 'compile'):
    try:
        netG_DC = torch.compile(netG_DC)
        netD_DC = torch.compile(netD_DC)
        print("DCGAN models compiled with torch.compile() ✓")
    except Exception as e:
        print(f"torch.compile skipped: {e}")

print("DCGAN Architecture Ready.")


## Step 5: DCGAN Training Loop

I train the DCGAN model using Binary Cross-Entropy (`BCEWithLogitsLoss`) and PyTorch AMP (`GradScaler`).
Losses are logged after each batch, and sample face grids are generated at the end of every epoch.


In [ ]:
criterion     = nn.BCEWithLogitsLoss()
optimizerD_DC = optim.Adam(netD_DC.parameters(), lr=LR_DCGAN, betas=(BETA1, 0.999))
optimizerG_DC = optim.Adam(netG_DC.parameters(), lr=LR_DCGAN, betas=(BETA1, 0.999))

scaler_D_DC = torch.amp.GradScaler('cuda', enabled=USE_AMP)
scaler_G_DC = torch.amp.GradScaler('cuda', enabled=USE_AMP)

DCGAN_CKPT  = CKPT_DIR / "dcgan_checkpoint.pth"

start_epoch = 0
G_losses_DC, D_losses_DC = [], []
best_loss_DC = float('inf')

if DCGAN_CKPT.exists():
    print(f"Loading DCGAN Checkpoint: {DCGAN_CKPT}")
    ckpt = torch.load(DCGAN_CKPT, map_location=device)
    start_epoch = ckpt['epoch'] + 1
    clean_sd_G = {k.replace('_orig_mod.', ''): v for k, v in ckpt['netG'].items()}
    clean_sd_D = {k.replace('_orig_mod.', ''): v for k, v in ckpt['netD'].items()}
    (netG_DC.module if isinstance(netG_DC, nn.DataParallel) else netG_DC).load_state_dict(clean_sd_G, strict=False)
    (netD_DC.module if isinstance(netD_DC, nn.DataParallel) else netD_DC).load_state_dict(clean_sd_D, strict=False)
    optimizerG_DC.load_state_dict(ckpt['optG'])
    optimizerD_DC.load_state_dict(ckpt['optD'])
    scaler_G_DC.load_state_dict(ckpt['scalerG'])
    scaler_D_DC.load_state_dict(ckpt['scalerD'])
    G_losses_DC = ckpt['G_losses']
    D_losses_DC = ckpt['D_losses']
    best_loss_DC = ckpt.get('best_loss', float('inf'))
    print(f"Resumed DCGAN Training from Epoch {start_epoch + 1}/{NUM_EPOCHS}")


def train_dcgan_epoch(epoch):
    netG_DC.train()
    netD_DC.train()
    
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    
    pbar = tqdm(dataloader, desc=f"DCGAN Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for i, data in enumerate(pbar):
        real_cpu = data.to(device, non_blocking=True)
        if USE_CHANNELS_LAST:
            real_cpu = real_cpu.to(memory_format=torch.channels_last)
        b_size = real_cpu.size(0)

        # ── Train Discriminator (with AMP FP16) ──────────────
        optimizerD_DC.zero_grad(set_to_none=True)
        label_real = torch.full((b_size,), 0.9, dtype=torch.float, device=device) # Label smoothing
        label_fake = torch.full((b_size,), 0.0, dtype=torch.float, device=device)
        
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out_real  = netD_DC(real_cpu).view(-1)
            errD_real = criterion(out_real, label_real)
        scaler_D_DC.scale(errD_real).backward()

        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake  = netG_DC(noise)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out_fake  = netD_DC(fake.detach()).view(-1)
            errD_fake = criterion(out_fake, label_fake)
        scaler_D_DC.scale(errD_fake).backward()

        scaler_D_DC.step(optimizerD_DC)
        scaler_D_DC.update()
        errD = errD_real + errD_fake

        # ── Train Generator (with AMP FP16) ──────────────────
        optimizerG_DC.zero_grad(set_to_none=True)
        label_gen = torch.full((b_size,), 1.0, dtype=torch.float, device=device)
        
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out_g = netD_DC(fake).view(-1)
            errG  = criterion(out_g, label_gen)

        scaler_G_DC.scale(errG).backward()
        scaler_G_DC.step(optimizerG_DC)
        scaler_G_DC.update()

        loss_g_val = errG.item()
        loss_d_val = errD.item()
        G_losses_DC.append(loss_g_val)
        D_losses_DC.append(loss_d_val)
        epoch_g_loss += loss_g_val
        epoch_d_loss += loss_d_val

        # Immediate Local Reference Cleanup
        del real_cpu, noise, fake, out_real, out_fake, errD_real, errD_fake, errD, errG

        if i % 100 == 0:
            pbar.set_postfix({'D_Loss': f"{loss_d_val:.4f}", 'G_Loss': f"{loss_g_val:.4f}"})
            
        if i % 500 == 0 and torch.cuda.is_available():
            print(f"[DCGAN Epoch {epoch+1} Batch {i}/{len(dataloader)}] "
                  f"GPU Alloc: {torch.cuda.memory_allocated()/(1024**2):.0f}MB | "
                  f"Reserved: {torch.cuda.memory_reserved()/(1024**2):.0f}MB")

    return epoch_g_loss / len(dataloader), epoch_d_loss / len(dataloader)


print("Starting DCGAN Training Loop...")
for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    avg_g, avg_d = train_dcgan_epoch(epoch)
    elapsed = time.time() - t0

    # End-of-Epoch Image Generation & Async Save
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            fake_grid = netG_DC(fixed_noise).detach().cpu()
    img_path = IMAGES_DIR / f"dcgan_epoch_{epoch+1:02d}.png"
    save_image_async(fake_grid, img_path)

    # Checkpoint & Best Model Saving
    raw_G = netG_DC.module if isinstance(netG_DC, nn.DataParallel) else netG_DC
    raw_D = netD_DC.module if isinstance(netD_DC, nn.DataParallel) else netD_DC
    
    ckpt_data = {
        'epoch': epoch,
        'netG': raw_G.state_dict(),
        'netD': raw_D.state_dict(),
        'optG': optimizerG_DC.state_dict(),
        'optD': optimizerD_DC.state_dict(),
        'scalerG': scaler_G_DC.state_dict(),
        'scalerD': scaler_D_DC.state_dict(),
        'G_losses': G_losses_DC,
        'D_losses': D_losses_DC,
        'best_loss': min(best_loss_DC, avg_g),
    }
    torch.save(ckpt_data, DCGAN_CKPT)
    
    if avg_g < best_loss_DC:
        best_loss_DC = avg_g
        torch.save(raw_G.state_dict(), MODELS_DIR / 'netG_DC_best.pth')

    # Memory Cleanup at Epoch End Only
    gc.collect()
    torch.cuda.empty_cache()

    clear_output(wait=True)
    print(f"DCGAN Epoch {epoch+1}/{NUM_EPOCHS} | Time: {elapsed:.1f}s | Avg_G: {avg_g:.4f} | Avg_D: {avg_d:.4f}")
    print(get_system_stats())
    display(Image(filename=str(img_path)))

# Save Final DCGAN Model & Metrics
raw_G = netG_DC.module if isinstance(netG_DC, nn.DataParallel) else netG_DC
torch.save(raw_G.state_dict(), MODELS_DIR / 'netG_DC_final.pth')
with open(METRICS_DIR / 'dcgan_metrics.json', 'w') as f:
    json.dump({'G_loss': G_losses_DC, 'D_loss': D_losses_DC}, f)

print("✅ DCGAN Training Complete!")


## Step 6: WGAN-GP Training Loop

I train the WGAN-GP model using the Wasserstein distance and `N_CRITIC = 1`.
Convolutions run in FP16 for speed, while the Gradient Penalty calculation is isolated in FP32 (`autocast(enabled=False)`) for numerical stability.


In [ ]:
del netG_DC, netD_DC, optimizerG_DC, optimizerD_DC, scaler_G_DC, scaler_D_DC
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared ✓")
print(get_system_stats())


## Step 7: Evaluate Generated Face Quality (FID Score)

To evaluate the models objectively, I calculate the Fréchet Inception Distance (FID) score.
A lower FID score means the generated faces are more realistic and closer to the distribution of real CelebA images.


In [ ]:
class WGANGP_Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(NZ,    NGF*8, 4, 1, 0, bias=False), nn.BatchNorm2d(NGF*8), nn.ReLU(False),
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF*4), nn.ReLU(False),
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False), nn.BatchNorm2d(NGF*2), nn.ReLU(False),
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False), nn.BatchNorm2d(NGF),   nn.ReLU(False),
            nn.ConvTranspose2d(NGF,   3,     4, 2, 1, bias=False), nn.Tanh(),
        )
    def forward(self, x): return self.main(x)


class WGANGP_Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3,     NDF,   4, 2, 1, bias=False),                               nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(NDF,   NDF*2, 4, 2, 1, bias=False), nn.InstanceNorm2d(NDF*2, affine=True),  nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False), nn.InstanceNorm2d(NDF*4, affine=True),  nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False), nn.InstanceNorm2d(NDF*8, affine=True),  nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(NDF*8, 1,     4, 1, 0, bias=False), # No Sigmoid
        )
    def forward(self, x): return self.main(x)


netG_W = WGANGP_Generator().to(device)
netC_W = WGANGP_Critic().to(device)

if USE_CHANNELS_LAST:
    # Only convert Critic (Conv2d); ConvTranspose2d in Generator has Inductor stride bugs
    netC_W = netC_W.to(memory_format=torch.channels_last)

if USE_MULTI_GPU and ngpu > 1:
    netG_W = nn.DataParallel(netG_W, list(range(ngpu)))
    netC_W = nn.DataParallel(netC_W, list(range(ngpu)))

netG_W.apply(weights_init)
netC_W.apply(weights_init)

# Note: WGAN-GP is kept in Eager Mode (not compiled with torch.compile) because Gradient Penalty
# requires double backward (create_graph=True), which torch.compile/aot_autograd does not support.
print("WGAN-GP models ready in Eager Mode (required for Gradient Penalty double backward) ✓")

print("WGAN-GP Architecture Ready.")


## Step 8: Quantitative Comparison Table

Here I print a summary table comparing DCGAN vs. WGAN-GP across FID score, SSIM, PSNR, training time, and stability.


In [ ]:
optimizerC_W = optim.Adam(netC_W.parameters(), lr=LR_WGAN, betas=(0.0, 0.9))
optimizerG_W = optim.Adam(netG_W.parameters(), lr=LR_WGAN, betas=(0.0, 0.9))

WGANGP_CKPT = CKPT_DIR / "wgangp_checkpoint.pth"

start_epoch_w = 0
G_losses_W, C_losses_W = [], []
best_loss_W = float('inf')

if WGANGP_CKPT.exists():
    print(f"Loading WGAN-GP Checkpoint: {WGANGP_CKPT}")
    ckpt_w = torch.load(WGANGP_CKPT, map_location=device)
    start_epoch_w = ckpt_w['epoch'] + 1
    clean_sd_G_w = {k.replace('_orig_mod.', ''): v for k, v in ckpt_w['netG'].items()}
    clean_sd_C_w = {k.replace('_orig_mod.', ''): v for k, v in ckpt_w['netC'].items()}
    (netG_W.module if isinstance(netG_W, nn.DataParallel) else netG_W).load_state_dict(clean_sd_G_w, strict=False)
    (netC_W.module if isinstance(netC_W, nn.DataParallel) else netC_W).load_state_dict(clean_sd_C_w, strict=False)
    optimizerG_W.load_state_dict(ckpt_w['optG'])
    optimizerC_W.load_state_dict(ckpt_w['optC'])
    G_losses_W = ckpt_w['G_losses']
    C_losses_W = ckpt_w['C_losses']
    best_loss_W = ckpt_w.get('best_loss', float('inf'))
    print(f"Resumed WGAN-GP Training from Epoch {start_epoch_w + 1}/{NUM_EPOCHS}")


def compute_gradient_penalty(critic, real_samples, fake_samples, dev):
    """Memory-efficient Gradient Penalty with autograd matching shapes."""
    b = real_samples.size(0)
    alpha = torch.rand(b, 1, 1, 1, device=dev)
    if USE_CHANNELS_LAST:
        alpha = alpha.to(memory_format=torch.channels_last)
    
    interpolates = (alpha * real_samples + ((1.0 - alpha) * fake_samples)).requires_grad_(True)
    
    d_interpolates = critic(interpolates)

    grad_outputs = torch.ones_like(d_interpolates, device=dev)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(b, -1)
    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean() * LAMBDA_GP
    return gp


def train_wgan_epoch(epoch):
    netG_W.train()
    netC_W.train()
    
    epoch_c_loss = 0.0
    epoch_g_loss = 0.0
    
    pbar = tqdm(dataloader, desc=f"WGAN-GP Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for i, data in enumerate(pbar):
        real_cpu = data.to(device, non_blocking=True)
        if USE_CHANNELS_LAST:
            real_cpu = real_cpu.to(memory_format=torch.channels_last)
        b_size = real_cpu.size(0)

        # ── Train Critic N_CRITIC times (FP16 Tensor Cores + FP32 Gradient Penalty) ──
        for _ in range(N_CRITIC):
            optimizerC_W.zero_grad(set_to_none=True)
            noise_c = torch.randn(b_size, NZ, 1, 1, device=device)
            
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                fake_c   = netG_W(noise_c).detach()
                real_val = netC_W(real_cpu)
                fake_val = netC_W(fake_c)
            
            with torch.autocast(device_type='cuda', enabled=False):
                gp = compute_gradient_penalty(netC_W, real_cpu.float(), fake_c.float(), device)
            
            errC = -real_val.mean() + fake_val.mean() + gp

            errC.backward()
            optimizerC_W.step()
            del noise_c, fake_c, real_val, fake_val, gp

        # ── Train Generator once (FP16 Tensor Cores) ─────────────
        optimizerG_W.zero_grad(set_to_none=True)
        noise_g = torch.randn(b_size, NZ, 1, 1, device=device)
        
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            gen_fake = netG_W(noise_g)
            errG     = -netC_W(gen_fake).mean()

        errG.backward()
        optimizerG_W.step()

        loss_c_val = errC.item()
        loss_g_val = errG.item()
        C_losses_W.append(loss_c_val)
        G_losses_W.append(loss_g_val)
        epoch_c_loss += loss_c_val
        epoch_g_loss += loss_g_val

        del real_cpu, noise_g, gen_fake, errC, errG

        if i % 100 == 0:
            pbar.set_postfix({'C_Loss': f"{loss_c_val:.4f}", 'G_Loss': f"{loss_g_val:.4f}"})
            
        if i % 500 == 0 and torch.cuda.is_available():
            print(f"[WGAN-GP Epoch {epoch+1} Batch {i}/{len(dataloader)}] "
                  f"GPU Alloc: {torch.cuda.memory_allocated()/(1024**2):.0f}MB | "
                  f"Reserved: {torch.cuda.memory_reserved()/(1024**2):.0f}MB")

    return epoch_g_loss / len(dataloader), epoch_c_loss / len(dataloader)


print("Starting WGAN-GP Training Loop...")
for epoch in range(start_epoch_w, NUM_EPOCHS):
    t0 = time.time()
    avg_g, avg_c = train_wgan_epoch(epoch)
    elapsed = time.time() - t0

    # End-of-Epoch Image Generation & Async Save
    with torch.no_grad():
        fake_grid = netG_W(fixed_noise).detach().cpu()
    img_path = IMAGES_DIR / f"wgangp_epoch_{epoch+1:02d}.png"
    save_image_async(fake_grid, img_path)

    raw_G = netG_W.module if isinstance(netG_W, nn.DataParallel) else netG_W
    raw_C = netC_W.module if isinstance(netC_W, nn.DataParallel) else netC_W

    ckpt_w_data = {
        'epoch': epoch,
        'netG': raw_G.state_dict(),
        'netC': raw_C.state_dict(),
        'optG': optimizerG_W.state_dict(),
        'optC': optimizerC_W.state_dict(),
        'G_losses': G_losses_W,
        'C_losses': C_losses_W,
        'best_loss': min(best_loss_W, avg_c),
    }
    torch.save(ckpt_w_data, WGANGP_CKPT)

    if avg_c < best_loss_W:
        best_loss_W = avg_c
        torch.save(raw_G.state_dict(), MODELS_DIR / 'netG_WGANGP_best.pth')

    # Memory Cleanup at Epoch End Only
    gc.collect()
    torch.cuda.empty_cache()

    clear_output(wait=True)
    print(f"WGAN-GP Epoch {epoch+1}/{NUM_EPOCHS} | Time: {elapsed:.1f}s | Avg_G: {avg_g:.4f} | Avg_C: {avg_c:.4f}")
    print(get_system_stats())
    display(Image(filename=str(img_path)))

# Save Final Models & Metrics
raw_G = netG_W.module if isinstance(netG_W, nn.DataParallel) else netG_W
torch.save(raw_G.state_dict(), MODELS_DIR / 'netG_WGANGP_final.pth')
with open(METRICS_DIR / 'wgangp_metrics.json', 'w') as f:
    json.dump({'G_loss': G_losses_W, 'C_loss': C_losses_W}, f)

print("✅ WGAN-GP Training Complete!")


## Step 9: Side-by-Side Visual Comparison

In this section, I display a side-by-side comparison grid of:
1. Real CelebA human faces.
2. Faces generated by DCGAN after 50 epochs.
3. Faces generated by WGAN-GP after 50 epochs.


In [ ]:
# ── Loss Curves Plot ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].set_title("DCGAN-SN Losses (Spectral Normalization)")
axes[0].plot(G_losses_DC, label="Generator",     alpha=0.6, color='orange')
axes[0].plot(D_losses_DC, label="Discriminator", alpha=0.6, color='royalblue')
axes[0].set_xlabel("Iterations"); axes[0].set_ylabel("Loss"); axes[0].legend()

axes[1].set_title("WGAN-GP Losses (Wasserstein Distance Estimate)")
axes[1].plot(G_losses_W, label="Generator", alpha=0.6, color='orange')
axes[1].plot(C_losses_W, label="Critic",    alpha=0.6, color='seagreen')
axes[1].set_xlabel("Iterations"); axes[1].set_ylabel("Wasserstein Distance"); axes[1].legend()

plt.tight_layout()
plt.savefig(str(IMAGES_DIR / 'loss_comparison.png'), bbox_inches='tight')
plt.close('all')
display(Image(filename=str(IMAGES_DIR / 'loss_comparison.png')))

# ── Image Comparison Plot ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
titles    = ["Real CelebA Faces", f"DCGAN (Epoch {NUM_EPOCHS})", f"WGAN-GP (Epoch {NUM_EPOCHS})"]
imgs      = [
    np.transpose(vutils.make_grid(real_batch_cpu[:64], padding=2, normalize=True).numpy(), (1, 2, 0)),
    PILImage.open(IMAGES_DIR / f"dcgan_epoch_{NUM_EPOCHS:02d}.png"),
    PILImage.open(IMAGES_DIR / f"wgangp_epoch_{NUM_EPOCHS:02d}.png"),
]
for ax, title, img in zip(axes, titles, imgs):
    ax.axis("off"); ax.set_title(title); ax.imshow(img)

plt.tight_layout()
plt.savefig(str(IMAGES_DIR / 'image_comparison.png'), bbox_inches='tight')
plt.close('all')
display(Image(filename=str(IMAGES_DIR / 'image_comparison.png')))
print("Comparison Plots Successfully Saved ✓")


## Step 10: Package Results and Download

Finally, I compress the saved model weights, generated face grids, and metrics into a `.zip` archive so they can be easily downloaded from Kaggle.


In [ ]:
zip_name = '/kaggle/working/Advanced_GAN_Results'
shutil.make_archive(zip_name, 'zip', str(OUTPUT_DIR))
print(f"\nAll results compressed to: {zip_name}.zip")
print("Download from Kaggle Output Sidebar (/kaggle/working/).")
